In [ ]:
import geopandas as gpd
import pandas as pd
import seaborn as sb
import matplotlib.pyplot as plt
import numpy as np
from shapely.geometry import Point
from shapely.geometry.polygon import Polygon
from matplotlib.colors import ListedColormap

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
divipola_df=pd.read_csv('/content/drive/MyDrive/eco2026/Divipola.csv')
divipola_df.info()

In [ ]:
divipola_df_renamed = divipola_df.rename(columns={
    'Código Departamento': 'cod_departamento',
    'Código Municipio': 'cod_municipio',
    'Nombre Departamento': 'nombre_depto',
    'Nombre Municipio': 'nombre_mpio'
})
divipola_df_renamed.to_csv('/content/drive/MyDrive/eco2026/Divipola_renamed.csv', index=False)

In [ ]:


municipios_por_departamento_df = divipola_df_renamed.groupby('cod_departamento')['cod_municipio'].nunique().reset_index()
municipios_por_departamento_df_sorted = municipios_por_departamento_df.sort_values(by='cod_departamento')

display(municipios_por_departamento_df_sorted)

In [ ]:
mcpos_depto = divipola_df_renamed.groupby('nombre_depto')['cod_municipio'].nunique().reset_index()
mcpos_depto.rename(columns={'cod_municipio': 'Numero_Municipios'}, inplace=True)
mcpos_depto = mcpos_depto.sort_values(by='nombre_depto')

display(mcpos_depto)

In [ ]:
unique_departamentos = divipola_df_renamed[['nombre_depto', 'cod_departamento']].drop_duplicates()
display(unique_departamentos.sort_values(by='cod_departamento'))

In [ ]:
divipola_json=pd.read_json('/content/drive/MyDrive/eco2026/Divipola_Municipios.json')
divipola_parsedj = pd.json_normalize(divipola_json['resultado'])

In [ ]:
# Ensure divipola_parsed has the correct types and columns from previous steps

# Convert 'CODIGO_DEPARTAMENTO' to numeric (if not already numeric or contains non-numeric strings)
# Using a temporary copy to avoid SettingWithCopyWarning if divipola_parsed is a slice
divipola_parsedj['CODIGO_DEPARTAMENTO'] = pd.to_numeric(divipola_parsedj['CODIGO_DEPARTAMENTO'], errors='coerce')

# Drop 'CODIGO_MUNICIPIO' if it exists (it was requested to be removed previously)
if 'CODIGO_MUNICIPIO' in divipola_parsedj.columns:
    divipola_parsedj.drop(columns=['CODIGO_MUNICIPIO'], inplace=True)

# Create 'COD_MUNICIPIO' from CODIGO_DPTO_MPIO (last 3 digits), converting to numeric
# This ensures the column exists for subsequent operations
divipola_parsedj['COD_MUNICIPIO'] = pd.to_numeric(divipola_parsedj['CODIGO_DPTO_MPIO'].str[-3:], errors='coerce')

# Now proceed with the current request: renaming and grouping
divipola_renamedj = divipola_parsedj.rename(columns={
    'CODIGO_DEPARTAMENTO': 'cod_departamento',
    'COD_MUNICIPIO': 'cod_municipio'
})

# Count the number of municipalities per department using the renamed columns
municipios_por_departamento_df = divipola_renamedj.groupby('cod_departamento')['cod_municipio'].nunique().reset_index()

# Rename the count column for clarity
municipios_por_departamento_df.rename(columns={'cod_municipio': 'Numero_Municipios'}, inplace=True)

# Sort the result by 'cod_departamento'
municipios_por_departamento_df_sorted2 = municipios_por_departamento_df.sort_values(by='cod_departamento')

# Display the result
display(municipios_por_departamento_df_sorted2)

In [ ]:
divipola_df_renamed.info()

## Comparación de Conteo de Municipios por Departamento

A continuación, se comparará el número de municipios por departamento entre `municipios_por_departamento_df_sorted2` y `municipios_por_departamento_df_sorted` para identificar cualquier discrepancia.

In [ ]:
# Merge the two dataframes for comparison
comparison_of_counts = pd.merge(
    municipios_por_departamento_df_sorted,
    municipios_por_departamento_df_sorted2,
    on='cod_departamento',
    suffixes=('_df1', '_df2'),
    how='outer'
)

# Identify discrepancies where the number of municipalities differs
discrepancies = comparison_of_counts[comparison_of_counts['Numero_Municipios_df1'] != comparison_of_counts['Numero_Municipios_df2']]

if discrepancies.empty:
    print("No se encontraron diferencias en el número de municipios por departamento entre los dos dataframes.")
else:
    print("Departamentos con diferencias en el número de municipios:")
    display(discrepancies)


In [ ]:
mpios_file='https://opendata.arcgis.com/api/v3/datasets/623a71c7f5c94bada0416879df0effe4_0/downloads/data?format=shp&spatialRefId=4326&where=1%3D1'
mpios=gpd.read_file(mpios_file)

In [ ]:
mpios.info()

In [ ]:
mpios.drop(['OBJECTID_1','OBJECTID'], axis=1, inplace=True)

In [ ]:
mpios.info()

In [ ]:
# Crear mapa de colores, para distinguir los departamentos de manera única

cmap1 = plt.cm.tab20c
cmap2 = plt.cm.tab20b
colors1 = cmap1(np.linspace(0., 1, 128))
colors2 = cmap2(np.linspace(0, 1, 128))
colors_combined = np.vstack((colors1, colors2))
cmap_combined = ListedColormap(colors_combined, name='tab20combined')

In [ ]:
mpios.plot(figsize=(10,10), cmap=cmap_combined,column='DPTO_CCDGO')

## Validación de los datos de la DIVIPOLA oficial vs el mapa
Se validan los datos oficiales de los códigos de municipio y departamento vs los datos del mapa, para tener una base limpia y confirmada, tanto de los datos tabulares, como de los datos geográficos

### Validación de Departamentos
Cantidad, Nombres
Homogenizar Nombres Y validar existencia

In [ ]:
unique_dpto_mpio = mpios[['DPTO_CCDGO']].drop_duplicates().sort_values(by='DPTO_CCDGO').count()
print(unique_dpto_mpio)

In [ ]:
unique_dpto_divipola = divipola_df[['Código Departamento']].drop_duplicates().sort_values(by='Código Departamento').count()
print(unique_dpto_mpio)

In [ ]:
unique_dpto_nombre = mpios[['DEPTO','DPTO_CCDGO']].drop_duplicates().sort_values(by='DEPTO')
print(unique_dpto_nombre)

In [ ]:
unique_dpto_divinombre = divipola_df[['Nombre Departamento','Código Departamento']].drop_duplicates().sort_values(by='Nombre Departamento')
print(unique_dpto_divinombre)

In [ ]:
mpios['DPTO_CCDGO'] = pd.to_numeric(mpios['DPTO_CCDGO'], errors='coerce')
mpios.info()

In [ ]:
print(mpios[['DEPTO','DPTO_CCDGO']].drop_duplicates().sort_values(by='DEPTO'))

## Homogenizar nombres en un dataset
Tomando como base Divipola, se normalizar el nombre en el dataset de mpios, que es el que contiene la informacion geografica, necesaria para hacer algun tipo plot con los datos

In [ ]:
# Create a mapping dictionary from divipola_df (Código Departamento to Nombre Departamento)
# Ensure unique pairs are used to avoid InvalidIndexError
dpto_name_mapping = divipola_df.drop_duplicates(subset=['Código Departamento', 'Nombre Departamento']).set_index('Código Departamento')['Nombre Departamento']

# Apply this mapping to the 'DEPTO' column in mpios based on 'DPTO_CCDGO'
mpios['DEPTO'] = mpios['DPTO_CCDGO'].map(dpto_name_mapping)

# Display unique department names and codes from mpios to verify the update
print(mpios[['DEPTO', 'DPTO_CCDGO']].drop_duplicates().sort_values(by='DPTO_CCDGO'))

## Comparación de Conteo de Municipios por Departamento

Ahora que los nombres de los departamentos en `mpios` han sido homogeneizados, vamos a contar el número de municipios por departamento en ambos dataframes (`mpios` y `divipola_df`) y compararemos los resultados.

In [ ]:
# Count unique municipalities per department in mpios GeoDataFrame
mpios_municipality_counts = mpios.groupby('DEPTO')['MPIO_CCDGO'].nunique().reset_index()
mpios_municipality_counts.rename(columns={'MPIO_CCDGO': 'Mpios_Count_Map'}, inplace=True)

# Count unique municipalities per department in divipola_df DataFrame
divipola_municipality_counts = divipola_df.groupby('Nombre Departamento')['Código Municipio'].nunique().reset_index()
divipola_municipality_counts.rename(columns={'Código Municipio': 'Mpios_Count_Divipola'}, inplace=True)

# Merge the two counts for comparison
comparison_df = pd.merge(
    mpios_municipality_counts,
    divipola_municipality_counts,
    left_on='DEPTO',
    right_on='Nombre Departamento',
    how='outer'
)

# Drop the redundant 'Nombre Departamento' column and reorder for better readability
comparison_df.drop(columns=['Nombre Departamento'], inplace=True)
comparison_df = comparison_df[['DEPTO', 'Mpios_Count_Map', 'Mpios_Count_Divipola']]

# Display the comparison table, sorting by department name
display(comparison_df.sort_values(by='DEPTO'))

### Departamentos con Discrepancias en el Conteo de Municipios

A continuación, se muestra una tabla con los departamentos donde el número de municipios difiere entre los dos conjuntos de datos (`mpios` y `divipola_df`).

In [ ]:
# Identify departments where municipality counts differ
discrepancy_df = comparison_df[comparison_df['Mpios_Count_Map'] != comparison_df['Mpios_Count_Divipola']]

# Display the departments with discrepancies
display(discrepancy_df.sort_values(by='DEPTO'))

In [ ]:
divipola_geo = gpd.read_file('/content/drive/MyDrive/eco2026/DivipolaGeo.gpkg')
divipola_geo.info()

In [ ]:
divipola_geo = gpd.read_file('/content/drive/MyDrive/eco2026/DivipolaGeo.gpkg')
divipola_geo['MpCodigo'] = pd.to_numeric(divipola_geo['MpCodigo'], errors='coerce')
divipola_geo.rename(columns={'MpCodigo': 'cod_municipio', 'Depto': 'nombre_depto'}, inplace=True)
divipola_geo['nombre_depto'] = divipola_geo['nombre_depto'].str.upper()

display(divipola_geo.head())

In [ ]:
# Convert 'nombre_depto' in divipola_df_renamed to uppercase for consistent merging
divipola_df_renamed['nombre_depto_upper'] = divipola_df_renamed['nombre_depto'].str.upper()

# Create a mapping DataFrame from divipola_df_renamed for cod_departamento
departamento_code_mapping = divipola_df_renamed[['nombre_depto_upper', 'cod_departamento']].drop_duplicates()

# Merge divipola_geo with the mapping DataFrame
divipola_geo = pd.merge(
    divipola_geo,
    departamento_code_mapping,
    left_on='nombre_depto',
    right_on='nombre_depto_upper',
    how='left'
)

# Drop the auxiliary 'nombre_depto_upper' column from divipola_geo as it's redundant after merge
divipola_geo.drop(columns=['nombre_depto_upper'], inplace=True)

display(divipola_geo.head())

In [ ]:
municipios_por_departamento_geo = divipola_geo.groupby('nombre_depto')['cod_municipio'].nunique().reset_index()
municipios_por_departamento_geo.rename(columns={'cod_municipio': 'Numero_Municipios'}, inplace=True)

display(municipios_por_departamento_geo.sort_values(by='nombre_depto'))